In [1]:
import subprocess
import os
from os.path import join
import pandas as pd
from datetime import datetime

import intake, intake_esm

from IPython.display import display, HTML

from urllib.parse import quote


import numpy as np
import json

In [ ]:
# Gather all information in txt file
data_path = '/scratchu/yrobin/BC2602/FORMAT/CORDEX5-Adjust/bias-adjusted-output/EUR-11/'
subprocess.call(f"ls -R {data_path} | perl -pe 's/(.*)_(.*)_(.*)_(.*)_(.*)_(.*)_(.*)_(.*).nc/\u$1,$2,$3,$4,$5,$6,$7,$8,$&/' > CORDEX_Adjust_catalog.txt", shell=True)

In [10]:
work_dir = "/home/user/These/cordex_htws_cc3d"
with open(join(work_dir,'Data','CORDEX_Adjust_catalog.txt'),'r') as file:
    lines = file.readlines()

In [11]:
lines

['/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjusted-input/EUR-11/:\n',
 'CLMcom\n',
 'CNRM\n',
 'DMI\n',
 'GERICS\n',
 'ICTP\n',
 'KNMI\n',
 'MOHC\n',
 'MPI-CSC\n',
 'SMHI\n',
 '\n',
 '/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjusted-input/EUR-11/CLMcom:\n',
 'MOHC-HadGEM2-ES\n',
 'MPI-M-MPI-ESM-LR\n',
 '\n',
 '/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjusted-input/EUR-11/CLMcom/MOHC-HadGEM2-ES:\n',
 'historical\n',
 'rcp45\n',
 'rcp85\n',
 '\n',
 '/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjusted-input/EUR-11/CLMcom/MOHC-HadGEM2-ES/historical:\n',
 'r1i1p1\n',
 '\n',
 '/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjusted-input/EUR-11/CLMcom/MOHC-HadGEM2-ES/historical/r1i1p1:\n',
 'CLMcom-CCLM4-8-17\n',
 '\n',
 '/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjusted-input/EUR-11/CLMcom/MOHC-HadGEM2-ES/historical/r1i1p1/CLMcom-CCLM4-8-17:\n',
 'RAW-NN\n',
 '\n',
 '/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjusted-input/EUR-11/CLMcom/MOHC-HadGEM2-ES/historical/r1i1p1/CLMcom-CCLM4-8-17/RAW-NN:\

In [12]:
columns = ['path','project','product','domain','driving_model','experiment','ensemble','rcm_model','rcm_version',
'time_frequency','variable','version','period_start','period_end','latest','correction','reference_dataset']
df = pd.DataFrame(data=None,columns=columns)

In [15]:
count=0
for i in range(len(lines)):
    if '.nc' in lines[i]:
        data = lines[i][:-1].split(",") # remove '\n'
        print(data)
        previous_line = lines[i-1][:-2] # Get previous line and remove ':\n'
        print(previous_line)
        df.loc[count,'path'] = previous_line+'/'+data[-1]
        df.loc[count,'driving_model'] = data[2]
        df.loc[count,'experiment'] = data[3]
        df.loc[count,'ensemble'] = data[4]
        df.loc[count,'rcm_model'] = data[5]
        df.loc[count,'variable'] = previous_line.split('/')[-2]#[:-6]
        df.loc[count,'version'] = previous_line.split('/')[-1]
        df.loc[count,'time_frequency'] = data[7]
        df.loc[count,'period_start'] = datetime.strptime(data[8].split("-")[0],'%Y%m%d')
        df.loc[count,'period_end'] = datetime.strptime(data[8].split("-")[1],'%Y%m%d')
        count+=1
df['project'] = 'CORDEX5-Adjust'
df['product'] = 'bias-adjusted-input'
df['domain'] = 'EUR-11'
df['rcm_version'] = 'BC2602'
df['latest'] = True
df['correction'] = 'ERA5 regrid'
df['reference_dataset'] = 'ERA5-1976-2005'

['Huss', 'EUR-11', 'MOHC-HadGEM2-ES', 'historical', 'r1i1p1', 'CLMcom-CCLM4-8-17', 'RAW-NN', 'day', '19500101-20051230', 'huss_EUR-11_MOHC-HadGEM2-ES_historical_r1i1p1_CLMcom-CCLM4-8-17_RAW-NN_day_19500101-20051230.nc']
/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjusted-input/EUR-11/CLMcom/MOHC-HadGEM2-ES/historical/r1i1p1/CLMcom-CCLM4-8-17/RAW-NN/day/huss/v20260512
['Pr', 'EUR-11', 'MOHC-HadGEM2-ES', 'historical', 'r1i1p1', 'CLMcom-CCLM4-8-17', 'RAW-NN', 'day', '19500101-20051230', 'pr_EUR-11_MOHC-HadGEM2-ES_historical_r1i1p1_CLMcom-CCLM4-8-17_RAW-NN_day_19500101-20051230.nc']
/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjusted-input/EUR-11/CLMcom/MOHC-HadGEM2-ES/historical/r1i1p1/CLMcom-CCLM4-8-17/RAW-NN/day/pr/v20260512
['SfcWind', 'EUR-11', 'MOHC-HadGEM2-ES', 'historical', 'r1i1p1', 'CLMcom-CCLM4-8-17', 'RAW-NN', 'day', '19500101-20051230', 'sfcWind_EUR-11_MOHC-HadGEM2-ES_historical_r1i1p1_CLMcom-CCLM4-8-17_RAW-NN_day_19500101-20051230.nc']
/scratchu/tmandonnet/CORDEX5-Adjust/bias-adj

In [17]:
df.to_csv(join(work_dir,'CORDEX_ADJUST.csv'))

In [16]:
df

,path,project,product,domain,driving_model,experiment,ensemble,rcm_model,rcm_version,time_frequency,variable,version,period_start,period_end,latest,correction,reference_dataset
0,/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjus...,CORDEX5-Adjust,bias-adjusted-input,EUR-11,MOHC-HadGEM2-ES,historical,r1i1p1,CLMcom-CCLM4-8-17,BC2602,day,huss,v20260512,1950-01-01 00:00:00,2005-12-30 00:00:00,True,ERA5 regrid,ERA5-1976-2005
1,/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjus...,CORDEX5-Adjust,bias-adjusted-input,EUR-11,MOHC-HadGEM2-ES,historical,r1i1p1,CLMcom-CCLM4-8-17,BC2602,day,pr,v20260512,1950-01-01 00:00:00,2005-12-30 00:00:00,True,ERA5 regrid,ERA5-1976-2005
2,/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjus...,CORDEX5-Adjust,bias-adjusted-input,EUR-11,MOHC-HadGEM2-ES,historical,r1i1p1,CLMcom-CCLM4-8-17,BC2602,day,sfcWind,v20260512,1950-01-01 00:00:00,2005-12-30 00:00:00,True,ERA5 regrid,ERA5-1976-2005
3,/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjus...,CORDEX5-Adjust,bias-adjusted-input,EUR-11,MOHC-HadGEM2-ES,historical,r1i1p1,CLMcom-CCLM4-8-17,BC2602,day,tas,v20260512,1950-01-01 00:00:00,2005-12-30 00:00:00,True,ERA5 regrid,ERA5-1976-2005
4,/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjus...,CORDEX5-Adjust,bias-adjusted-input,EUR-11,MOHC-HadGEM2-ES,historical,r1i1p1,CLMcom-CCLM4-8-17,BC2602,day,tasmax,v20260512,1950-01-01 00:00:00,2005-12-30 00:00:00,True,ERA5 regrid,ERA5-1976-2005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjus...,CORDEX5-Adjust,bias-adjusted-input,EUR-11,ICHEC-EC-EARTH,rcp85,r1i1p1,SMHI-RCA4,BC2602,day,pr,v20260512,2006-01-01 00:00:00,2100-12-31 00:00:00,True,ERA5 regrid,ERA5-1976-2005
180,/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjus...,CORDEX5-Adjust,bias-adjusted-input,EUR-11,ICHEC-EC-EARTH,rcp85,r1i1p1,SMHI-RCA4,BC2602,day,sfcWind,v20260512,2006-01-01 00:00:00,2100-12-31 00:00:00,True,ERA5 regrid,ERA5-1976-2005
181,/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjus...,CORDEX5-Adjust,bias-adjusted-input,EUR-11,ICHEC-EC-EARTH,rcp85,r1i1p1,SMHI-RCA4,BC2602,day,tas,v20260512,2006-01-01 00:00:00,2100-12-31 00:00:00,True,ERA5 regrid,ERA5-1976-2005
182,/scratchu/tmandonnet/CORDEX5-Adjust/bias-adjus...,CORDEX5-Adjust,bias-adjusted-input,EUR-11,ICHEC-EC-EARTH,rcp85,r1i1p1,SMHI-RCA4,BC2602,day,tasmax,v20260512,2006-01-01 00:00:00,2100-12-31 00:00:00,True,ERA5 regrid,ERA5-1976-2005
